In [ ]:
from openai import OpenAI
import os 
api_key = os.getenv('api_key')
openai_client = OpenAI(api_key=api_key)

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
len(documents)

In [ ]:
from minsearch import Index 

index = Index(text_fields=['content'],
              keyword_fields=['filename'])
index.fit(documents)

In [ ]:
def search(question):
    boost_dict = {'filename': 1.0, 'content': 2.0}
    return index.search(question, boost_dict=boost_dict, num_results=5)

In [ ]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [ ]:
search_results = search(question)

In [ ]:
search_results

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('File name: ' + doc['filename'])
        lines.append('Content: ' +doc['content'])
        lines.append('')

    return '\n'.join(lines).strip()

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [ ]:

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [ ]:
model='gpt-5.4-mini'


In [ ]:
def build_prompt(query, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(
        question=query, context=context
    )

In [ ]:
def llm(prompt):
    input_messages = [
        {'role': 'developer', 'content': INSTRUCTIONS},
        {'role': 'user', 'content': prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=input_messages
    )

    return response

In [ ]:
search_results = search(question)
prompt = build_prompt(question, search_results)
response = llm(prompt)

In [ ]:
print(response.usage.input_tokens)

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
len(chunks)

In [ ]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields= ["filename"]
)

chunk_index.fit(chunks)

In [ ]:
def search(question):
    boost_dict = {'filename': 1.0, 'content': 2.0}
    return chunk_index.search(question, boost_dict=boost_dict, num_results=5)

In [ ]:
search_results = search(question)
prompt = build_prompt(question, search_results)
response = llm(prompt)

In [ ]:
print(response.usage.input_tokens)

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agent_tools.get_tools()

In [ ]:
AGENTIC_LOOP_INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

Make multiple searches.
"""


In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)



In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("api_key"))

llm_client = OpenAIClient(
    model="gpt-5.4-mini",
    client=client
)

In [ ]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=AGENTIC_LOOP_INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=llm_client)

In [ ]:
question = "How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(
    prompt=question,
    callback=callback,
)